# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

In [ ]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [1]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 16 procesach (rdzeniach)...
Zakończono w czasie 0.60s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [22]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def fetch_cat_fact(url):
    response = requests.get(url).json().get('fact')
    return response

start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    facts = list(executor.map(fetch_cat_fact, [CAT_API_URL] * 20))

facts = [fact for fact in facts]
print(f'Zajęło to: {time.time()-start}')
for fact in facts:
    print(fact)
    
# start = time.time()
# facts = [fetch_cat_fact(CAT_API_URL) for _ in range(20)]
# print(f"Zajeło to: {time.time() - start}")
# for fact in facts:
#     print(fact)

# Twój kod tutaj...

Zajęło to: 0.7142252922058105
A cat’s heart beats nearly twice as fast as a human heart, at 110 to 140 beats a minute.
Cats often overract to unexpected stimuli because of their extremely sensitive nervous system.
In Ancient Egypt, when a person's house cat passed away, the owner would shave their eyebrows to reflect their grief.
Statistics indicate that animal lovers in recent years have shown a preference for cats over dogs!
A cat’s eyesight is both better and worse than humans. It is better because cats can see in much dimmer light and they have a wider peripheral view. It’s worse because they don’t see color as well as humans do. Scientists believe grass appears red to cats.
A cat can’t climb head first down a tree because every claw on a cat’s paw points the same way. To get down from a tree, a cat must back down.
A sexually-active feral tom-cat \owns\" an area of about three square miles and \""sprays\"" to mark his territory with strong smelling urine."""
Cats control the outer 

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [50]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

input_queue = queue.Queue()
even_queue = queue.Queue()
odd_queue = queue.Queue()


def producent():
    for number in range(1, 21):
        input_queue.put(number)
        print(f"Producent dodał: {number}")
        time.sleep(0.05)

    input_queue.put(None)


def dispatcher():
    while True:
        item = input_queue.get()

        if item is None:
            even_queue.put(None)
            odd_queue.put(None)
            break

        if item % 2 == 0:
            even_queue.put(item)
        else:
            odd_queue.put(item)


def even_consumer():
    while True:
        item = even_queue.get()
        if item is None:
            break
        print(f"Konsument parzystych: {item}")


def odd_consumer():
    while True:
        item = odd_queue.get()
        if item is None:
            break
        print(f"Konsument nieparzystych: {item}")


producer_thread = threading.Thread(target=producent)
dispatcher_thread = threading.Thread(target=dispatcher)
even_thread = threading.Thread(target=even_consumer)
odd_thread = threading.Thread(target=odd_consumer)

producer_thread.start()
dispatcher_thread.start()
even_thread.start()
odd_thread.start()

producer_thread.join()
dispatcher_thread.join()
even_thread.join()
odd_thread.join()

print("Koniec programu")

Producent dodał: 1
Konsument nieparzystych: 1
Producent dodał: 2
Konsument parzystych: 2
Producent dodał: 3
Konsument nieparzystych: 3
Producent dodał: 4
Konsument parzystych: 4
Producent dodał: 5
Konsument nieparzystych: 5
Producent dodał: 6Konsument parzystych: 6

Producent dodał: 7
Konsument nieparzystych: 7
Producent dodał: 8
Konsument parzystych: 8
Producent dodał: 9
Konsument nieparzystych: 9
Producent dodał: 10
Konsument parzystych: 10
Producent dodał: 11Konsument nieparzystych: 11

Producent dodał: 12
Konsument parzystych: 12
Producent dodał: 13
Konsument nieparzystych: 13
Producent dodał: 14
Konsument parzystych: 14
Producent dodał: 15
Konsument nieparzystych: 15
Producent dodał: 16Konsument parzystych: 16

Producent dodał: 17
Konsument nieparzystych: 17
Producent dodał: 18
Konsument parzystych: 18
Producent dodał: 19
Konsument nieparzystych: 19
Producent dodał: 20
Konsument parzystych: 20
Koniec programu


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [64]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    cores = multiprocessing.cpu_count()
    
    print("Wieloprocesowość: ")
    print(f'Będziemy pracować na {cores} rdzeniach')
    

    numbers = range(1,10001)
    start = time.time()
    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, numbers)
    print(f'Trwało to: {(time.time()-start):.2f}')
    print(results[:3])


    print("Sekwencyjnie: ")
    start = time.time()
    results = [calculate_power_sum(n) for n in numbers]
    print(f'Sekwencyjnie trwało to: {(time.time()-start):.2f}')
    print(results[:3])

Wieloprocesowość: 
Będziemy pracować na 16 rdzeniach
Trwało to: 0.39
[100, 2535301200456458802993406410750, 773066281098016996554691694648431909053161283000]
Sekwencyjnie: 
Sekwencyjnie trwało to: 0.49
[100, 2535301200456458802993406410750, 773066281098016996554691694648431909053161283000]
